# Mnemonics — Eval 2: bge-reranker-v2-m3 + turn mode

**Başlamadan önce:** Runtime → Change runtime type → **T4 GPU**

Config: `--chunk-mode turn --temporal-aware --augment-preferences --candidate-k 50 --mode rerank`

CE: `BAAI/bge-reranker-v2-m3` (güçlü cross-encoder, turn chunk ile ilk test)

In [ ]:
# 1) Repo
!rm -rf /content/mnemonics
!git clone https://github.com/nakata-app/mnemonics.git /content/mnemonics
!git -C /content/mnemonics log --oneline -3

In [ ]:
# 2) Bagimliliklar
!pip install -q -e /content/mnemonics sentence-transformers numpy adaptmem 2>&1 | tail -5
print('Install OK')

In [ ]:
# 3) Dataset — longmemeval_s_cleaned.json'u sol panelden yukle (277 MB)
import os
DATA = '/content/longmemeval_s_cleaned.json'
size = os.path.getsize(DATA)
assert size > 100_000_000, f'Dataset cok kucuk: {size} byte — yukle!'
print(f'Dataset: {size/1e6:.1f} MB OK')

In [ ]:
# 4) Smoke test (5q)
import json, os, subprocess
os.makedirs('/content/results', exist_ok=True)
env = os.environ.copy()
env['LME_DATA'] = '/content/longmemeval_s_cleaned.json'
env['MNEMONICS_RERANK_MODEL'] = 'BAAI/bge-reranker-v2-m3'

subprocess.run([
    'python', 'benchmarks/longmemeval_eval.py',
    '--n', '5', '--mode', 'rerank',
    '--chunk-mode', 'turn',
    '--temporal-aware',
    '--augment-preferences',
    '--candidate-k', '50', '--seed', '42',
    '--out', '/tmp/smoke2.json',
], cwd='/content/mnemonics', env=env, check=True)

s = json.load(open('/tmp/smoke2.json'))['mnemonics_rerank']
print(f'=== SMOKE OK === R@1={s["R@1"]}')

In [ ]:
# 5) 100q eval
import json, subprocess
print('=== 100q — bge-reranker-v2-m3 + turn mode ===')

subprocess.run([
    'python', 'benchmarks/longmemeval_eval.py',
    '--n', '100', '--mode', 'rerank',
    '--chunk-mode', 'turn',
    '--temporal-aware',
    '--augment-preferences',
    '--candidate-k', '50', '--seed', '42',
    '--out', '/content/results/lme100_eval2.json',
    '--per-q-out', '/content/results/lme100_eval2_perq.json',
], cwd='/content/mnemonics', env=env, check=True)

r = json.load(open('/content/results/lme100_eval2.json'))['mnemonics_rerank']
print(f'\n100q → R@1={r["R@1"]:.3f}  R@5={r["R@5"]:.3f}  R@10={r["R@10"]:.3f}')
print(f'Baseline (lme-0.954):  R@1=0.954  R@5=0.988  R@10=0.994')

In [ ]:
# 6) 500q eval
import json, subprocess
print('=== 500q — bge-reranker-v2-m3 + turn mode ===')

subprocess.run([
    'python', 'benchmarks/longmemeval_eval.py',
    '--n', '500', '--mode', 'rerank',
    '--chunk-mode', 'turn',
    '--temporal-aware',
    '--augment-preferences',
    '--candidate-k', '50', '--seed', '42',
    '--out', '/content/results/lme500_eval2.json',
    '--per-q-out', '/content/results/lme500_eval2_perq.json',
], cwd='/content/mnemonics', env=env, check=True)

r = json.load(open('/content/results/lme500_eval2.json'))['mnemonics_rerank']
print(f'\n=== EVAL-2 FINAL 500q ===')
print(f'Config: bge-reranker-v2-m3 + turn + temporal + augment-preferences')
print(f'R@1={r["R@1"]:.3f}  R@5={r["R@5"]:.3f}  R@10={r["R@10"]:.3f}')
print(f'\nKarsilastirma:')
print(f'  lme-0.954: R@1=0.954  R@5=0.988  R@10=0.994')
print(f'  Hedef:     R@1=0.980  R@5=0.990  R@10=1.000')
delta = r["R@1"] - 0.954
print(f'  Delta R@1: {delta:+.3f}')
print('\nBy type:')
for qt in sorted(r['by_type']):
    b = r['by_type'][qt]
    print(f'  {qt:30} n={b["n"]:3}  R@1={b["R@1"]:.3f}')

In [ ]:
# 7) Indir
from google.colab import files
import os
for f in sorted(os.listdir('/content/results')):
    files.download(f'/content/results/{f}')
print('Bitti.')